In [1]:
import mediapipe as mp
import cv2
import numpy as np

In [2]:
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Make Basic Detection

In [3]:
# Getting video feed
cap = cv2.VideoCapture(0)
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False 

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True 
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass

        # Rendering
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2),
                                  )

        cv2.imshow('Raw Webcam Feed', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

[x: 0.47514445
y: 0.774999
z: -1.5189155
visibility: 0.979353
, x: 0.48770398
y: 0.715955
z: -1.4107513
visibility: 0.9590204
, x: 0.5104298
y: 0.7195022
z: -1.4121146
visibility: 0.9656513
, x: 0.5324602
y: 0.72369695
z: -1.4119618
visibility: 0.9681483
, x: 0.41266912
y: 0.7112059
z: -1.4393303
visibility: 0.9730575
, x: 0.38005626
y: 0.7150402
z: -1.4407375
visibility: 0.9809826
, x: 0.3504967
y: 0.722983
z: -1.4417845
visibility: 0.97867465
, x: 0.5355348
y: 0.77639747
z: -0.753111
visibility: 0.9852094
, x: 0.30615234
y: 0.790467
z: -0.88304025
visibility: 0.9723173
, x: 0.5138464
y: 0.85014665
z: -1.2538762
visibility: 0.9495999
, x: 0.42803115
y: 0.85910743
z: -1.3031018
visibility: 0.94743013
, x: 0.676706
y: 1.1324822
z: -0.34395757
visibility: 0.9369812
, x: 0.16339102
y: 1.2162551
z: -0.4936424
visibility: 0.7798415
, x: 0.8737682
y: 1.4099829
z: -0.58048326
visibility: 0.18310164
, x: 0.14307937
y: 1.7300442
z: -0.97236234
visibility: 0.06157875
, x: 0.700111
y: 1.1353387
z

# 2. Determining Joints

<img src="https://i.imgur.com/3j8BPdc.png" style="height:300px" >

In [4]:
landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]

x: 0.7822893
y: 1.1679636
z: -0.41860804
visibility: 0.9326013

In [5]:
landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]

x: 0.8710408
y: 1.5790187
z: -0.31093368
visibility: 0.10586486

# 3. Calculate Angles

In [6]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle >180.0:
        angle = 360-angle
        
    return angle 

In [7]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]


In [8]:
shoulder, elbow, wrist

([0.7822893261909485, 1.1679636240005493],
 [0.8710408210754395, 1.5790187120437622],
 [0.8565276861190796, 1.6889004707336426])

In [9]:
calculate_angle(shoulder, elbow, wrist)

160.29216398074158

In [10]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # or any width you prefer
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)  # or any height you prefer
stage = ""
counter = 0
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, 
                  min_tracking_confidence=0.5,
                  model_complexity=2,
                  static_image_mode=False,
                  smooth_landmarks=False,
                          ) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
      
        # Make detection
        results = pose.process(image)
    
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        frame = cv2.flip(frame, 1)
        
        # Extract landmarks
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            for i, landmark in enumerate(landmarks):
                print(f"Landmark {i}: x={landmark.x}, y={landmark.y}, z={landmark.z}")

            # Get coordinates
            angles_to_calculate = {
                "right_elbow_right_shoulder_right_hip": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                ],
                "left_elbow_left_shoulder_left_hip": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                ],
                "right_knee_mid_hip_left_knee": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                ],
                "right_hip_right_knee_right_ankle": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
                ],
                "left_hip_left_knee_left_ankle": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                ],
                "right_wrist_right_elbow_right_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                ],
                "left_wrist_left_elbow_left_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                ],
            }

            # Calculate and visualize angles
            angles = []
            for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
                angle = calculate_angle(points[0], points[1], points[2])
                angles.append(angle)
                cv2.putText(image, f'{angle_name}: {int(angle)}', 
                            (50, 100 + i * 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

            if angles[3] > 160 and angles[4] > 160:
                stage = "up"
            if angles[3] < 90 and angles[4] < 90 and stage == "up":
                stage = "down"
                counter += 1
        else:
            print("No landmarks detected in this frame.")

            
        # Render curl counter
        # Setup status box
        cv2.rectangle(image, (0,0), (225,73), (245,117,16), -1)
        
        # Rep data
        cv2.putText(image, 'REPS', (15,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), 
                    (10,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Stage data
        cv2.putText(image, 'STAGE', (65,12), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, 
                    (60,60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2), 
                                mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2) 
                                 )               
        
        # cv2.imshow('Mediapipe Feed', cv2.flip(image,1))
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

No landmarks detected in this frame.
No landmarks detected in this frame.
No landmarks detected in this frame.
No landmarks detected in this frame.
No landmarks detected in this frame.
No landmarks detected in this frame.
Landmark 0: x=0.5775127410888672, y=0.7691864967346191, z=-1.3185855150222778
Landmark 1: x=0.6014007329940796, y=0.6850852966308594, z=-1.233583688735962
Landmark 2: x=0.6208729147911072, y=0.6863279342651367, z=-1.2346032857894897
Landmark 3: x=0.6370827555656433, y=0.6898608207702637, z=-1.2343811988830566
Landmark 4: x=0.535459578037262, y=0.6842381954193115, z=-1.2573039531707764
Landmark 5: x=0.5138754844665527, y=0.6855088472366333, z=-1.2587367296218872
Landmark 6: x=0.4914395213127136, y=0.6886059045791626, z=-1.2596323490142822
Landmark 7: x=0.6466585397720337, y=0.7360742092132568, z=-0.6750054955482483
Landmark 8: x=0.4547699987888336, y=0.7533009052276611, z=-0.785215437412262
Landmark 9: x=0.612783670425415, y=0.8661238551139832, z=-1.0887311697006226
La

In [11]:
# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import nbimporter  # Import nbimporter

# from data import NeuralNetwork  
#  # Import your model class

# model = NeuralNetwork()  # Initialize the model
# model.load_state_dict(torch.load(r'D:\PROGRAMMING\BE proj24\data.pth'))  # Load the state dict
# model.eval() 

# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Get video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)  # or any width you prefer
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720) 
# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Prepare input data for your model
#             input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0)

#             # Predict next movement
#             with torch.no_grad():
#                 predicted_angles = model(input_tensor)
#                 predicted_angles = predicted_angles.squeeze().tolist()

#             # Display predicted movement
#             for i, angle in enumerate(predicted_angles):
#                 cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
#                             (50, 300 + i * 30),  # Adjust y-position to fit all angles
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA) 


#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                    mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

#     cap.release()
#     cv2.destroyAllWindows()


In [12]:
# import sys
# sys.path.append(r'D:\PROGRAMMING\New folder\Rehabilitation-System\data.ipynb')
# import nbimporter   # Import nbimporter
# from data import NeuralNetwork

In [13]:
# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import torch.nn as nn
# from my_main import LSTMModel

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load the model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Instantiate the model
# lstm_model = LSTMModel().to(device)

# # Load the state dictionary
# lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# # Set the model to evaluation mode
# lstm_model.eval()
# # Set up MediaPipe drawing and pose
# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Open video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# stage = ""
# counter = 0

# frame_data=[]
# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make pose detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks and calculate angles
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Define angle calculations (you can add other angles here)
#             angles_to_calculate = {
#                 "right_elbow_right_shoulder_right_hip": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                 ],
#                 "left_elbow_left_shoulder_left_hip": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                 ],
#                 "right_knee_mid_hip_left_knee": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
#                      (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                 ],
#                 "right_hip_right_knee_right_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
#                 ],
#                 "left_hip_left_knee_left_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
#                 ],
#                 "right_wrist_right_elbow_right_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                 ],
#                 "left_wrist_left_elbow_left_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                      landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                 ],
#             }

#             # Calculate angles and store them in the list for model input
#             angles = []
#             for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
#                 angle = calculate_angle(points[0], points[1], points[2])
#                 angles.append(angle)
#                 cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

#             # Store angles for multiple frames
#             frame_data.append(angles)

#             # Ensure there are 50 frames of data
#             if len(frame_data) > 50:
#                 frame_data.pop(0)

#             # Only make prediction if we have 50 frames
#             if len(frame_data) == 50:
#                 input_tensor = torch.tensor(frame_data, dtype=torch.float32).unsqueeze(0).to(device)

#                 # Predict next movement
#                 with torch.no_grad():
#                     predicted_angles = lstm_model(input_tensor).squeeze().tolist()

#                 # Display predicted next movement angles
#                 for i, angle in enumerate(predicted_angles):
#                     cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
#                                 (50, 300 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

#             # Rep counting logic
#             if angles[3] > 160 and angles[4] > 160:
#                 stage = "up"
#             if angles[3] < 90 and angles[4] < 90 and stage == "up":
#                 stage = "down"
#                 counter += 1

#         # Render rep counter and stage data
#         cv2.rectangle(image, (0, 0), (225, 73), (245, 117, 16), -1)
#         cv2.putText(image, 'REPS', (15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#         cv2.putText(image, str(counter), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)
#         cv2.putText(image, 'STAGE', (65, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
#         cv2.putText(image, stage, (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)

#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                   mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         # Display image
#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()

In [14]:


# import mediapipe as mp
# import cv2
# import numpy as np
# import torch
# import torch.nn as nn
# from my_main import LSTMModel

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Load the model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Instantiate the model
# lstm_model = LSTMModel().to(device)

# # Load the state dictionary
# lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# # Set the model to evaluation mode
# lstm_model.eval()
# # Set up MediaPipe drawing and pose
# mp_drawing = mp.solutions.drawing_utils
# mp_pose = mp.solutions.pose

# # Open video feed
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
# stage = ""
# counter = 0

# with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break

#         # Recolor image to RGB
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image.flags.writeable = False

#         # Make pose detection
#         results = pose.process(image)

#         # Recolor back to BGR
#         image.flags.writeable = True
#         image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

#         # Extract landmarks and calculate angles
#         if results.pose_landmarks:
#             landmarks = results.pose_landmarks.landmark

#             # Define angle calculations (you can add other angles here)
#             angles_to_calculate = {
#                 "right_elbow_right_shoulder_right_hip": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                 ],
#                 "left_elbow_left_shoulder_left_hip": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                 ],
#                 "right_knee_mid_hip_left_knee": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
#                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                 ],
#                 "right_hip_right_knee_right_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
#                 ],
#                 "left_hip_left_knee_left_ankle": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
#                 ],
#                 "right_wrist_right_elbow_right_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
#                 ],
#                 "left_wrist_left_elbow_left_shoulder": [
#                     [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
#                     [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
#                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
#                 ],
#             }

#             # Calculate current angles and store them in the list for model input
#             angles = []
#             for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
#                 angle = calculate_angle(points[0], points[1], points[2])
#                 angles.append(angle)
#                 cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

#             input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0).to(device)
#             input_tensor = input_tensor.repeat(64, 50, 1).to(device)
#             # Predict next movement
#             with torch.no_grad():
#                 predicted_angles = lstm_model(input_tensor).squeeze().tolist()

#             # Display predicted next movement angles
#             for i, angle in enumerate(predicted_angles):
#                 # Ensure angle is numeric
#                 if isinstance(angle, list):
#                     # If the angle is a list, flatten it and display each item
#                     for j, sub_angle in enumerate(angle):
#                         if isinstance(sub_angle, (int, float)):  # Ensure it's a number
#                             y_pos = 300 + (i * 30) + (j * 20)  # Adjusted y position for sub-angles
#                             cv2.putText(image, f'Predicted Angle {i+1}-{j+1}: {int(sub_angle)}', 
#                                         (300, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)
#                 elif isinstance(angle, (int, float)):  # Ensure the angle is numeric
#                     y_pos = 300 + i * 30  # Adjusted y position for main angles
#                     cv2.putText(image, f'Predicted Angle {i+1}: {int(angle)}', 
#                                 (300, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

#         # Render pose landmarks
#         mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
#                                   mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
#                                   mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

#         # Display image
#         cv2.imshow('Mediapipe Feed', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()

In [15]:


import mediapipe as mp
import cv2
import numpy as np
import torch
import torch.nn as nn
from my_main import LSTMModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the model
lstm_model = LSTMModel().to(device)

# Load the state dictionary
lstm_model.load_state_dict(torch.load("lstm_model.pth", map_location=device))

# Set the model to evaluation mode
lstm_model.eval()
# Set up MediaPipe drawing and pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Open video feed
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
stage = ""
counter = 0

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2, smooth_landmarks=False) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make pose detection
        results = pose.process(image)

        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Extract landmarks and calculate angles
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Define angle calculations (you can add other angles here)
            angles_to_calculate = {
                "right_elbow_right_shoulder_right_hip": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                ],
                "left_elbow_left_shoulder_left_hip": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                ],
                "right_knee_mid_hip_left_knee": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [(landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x) / 2,
                     (landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y + landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y) / 2],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                ],
                "right_hip_right_knee_right_ankle": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y],
                ],
                "left_hip_left_knee_left_ankle": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y],
                ],
                "right_wrist_right_elbow_right_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y],
                ],
                "left_wrist_left_elbow_left_shoulder": [
                    [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y],
                    [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, 
                     landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y],
                ],
            }

            # Calculate angles and store them in the list for model input
            angles = []
            for i, (angle_name, points) in enumerate(angles_to_calculate.items()):
                angle = calculate_angle(points[0], points[1], points[2])
                angles.append(angle)
                cv2.putText(image, f'{angle_name}: {int(angle)}', (50, 100 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

            input_tensor = torch.tensor(angles, dtype=torch.float32).unsqueeze(0).to(device)
            input_tensor = input_tensor.repeat(64, 50, 1).to(device)
            # Predict next movement
            with torch.no_grad():
                predicted_angles = lstm_model(input_tensor).squeeze().tolist()

            # Display predicted next movement angles
            for i, angle in enumerate(predicted_angles):
                if isinstance(angle, list):
                    for j, sub_angle in enumerate(angle):
                        if isinstance(sub_angle, (int, float)):
                            # Change color for predicted angles to blue
                            cv2.putText(image, f'Next Angle {i+1}-{j+1}: {int(sub_angle)}', 
                                        (50, 300 + (i * 30) + (j * 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2, cv2.LINE_AA)
                elif isinstance(angle, (int, float)):
                    # Change color for predicted angles to blue
                    cv2.putText(image, f'Next Angle {i+1}: {int(angle)}', 
                                (50, 300 + i * 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2, cv2.LINE_AA)


            # Rep counting logic
            if angles[3] > 160 and angles[4] > 160:
                stage = "up"
            if angles[3] < 90 and angles[4] < 90 and stage == "up":
                stage = "down"
                counter += 1

        # Render rep counter and stage data
        cv2.rectangle(image, (0, 0), (225, 73), (245, 117, 16), -1)
        cv2.putText(image, 'REPS', (15, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(image, 'STAGE', (65, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 2, cv2.LINE_AA)

        # Render pose landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2))

        # Display image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'my_main'

In [45]:
# import os
# print(os.path.isfile(r'D:\PROGRAMMING\New folder\Rehabilitation-System\data.ipynb'))

# Holistic

In [46]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [47]:
import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

cap = cv2.VideoCapture(0)

# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, 
                          min_tracking_confidence=0.5,
                          static_image_mode=False,
                          smooth_landmarks=True,
                          model_complexity=2
                            ) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )

        # 3. Left Hand
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )
                        
        cv2.imshow('Holistic Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [48]:
# SPINE ANGLE

In [49]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a = np.array(a)  # First point  
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates for back straightness check
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
            # Calculate angle for back straightness
            spine_angle = calculate_angle(shoulder, hip, knee)
            
            # Visualize spine angle
            cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
                        (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
            # Get foot positions for feet check
            left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
            right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
            # Check if feet are lifting off the ground
            if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
                cv2.putText(image, "Feet not planted!", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(image, "Feet planted", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
        except:
            pass
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
        cv2.imshow('Squat Form Detection', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


# Holistic

In [50]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic

In [51]:
import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_face_mesh = mp.solutions.face_mesh_connections  # Import face_mesh for FACE_CONNECTIONS

cap = cv2.VideoCapture(0)

# Initiate holistic model
with mp_holistic.Holistic(min_detection_confidence=0.5, 
                          min_tracking_confidence=0.5,
                          static_image_mode=False,
                          smooth_landmarks=True,
                          model_complexity=2
                            ) as holistic:
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor Feed
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False        
        
        # Make Detections
        results = holistic.process(image)
        
        # Recolor image back to BGR for rendering
        image.flags.writeable = True   
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # 1. Draw face landmarks
        mp_drawing.draw_landmarks(
            image, results.face_landmarks, mp_face_mesh.FACEMESH_TESSELATION,  # Use FACE_CONNECTIONS from face_mesh module
            mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
            mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
        )
        
        # 2. Right hand
        mp_drawing.draw_landmarks(
            image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
        )

        # 3. Left Hand
        mp_drawing.draw_landmarks(
            image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
        )

        # 4. Pose Detections
        mp_drawing.draw_landmarks(
            image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, 
            mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
            mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
        )
                        
        cv2.imshow('Holistic Webcam Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


In [52]:
# SPINE ANGLE

In [53]:
import cv2
import mediapipe as mp
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

def calculate_angle(a, b, c):
    a = np.array(a)  # First point  
    b = np.array(b)  # Midpoint
    c = np.array(c)  # End point
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=2) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        
        # Recolor image to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # Make detection
        results = pose.process(image)
        
        # Recolor back to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        try:
            landmarks = results.pose_landmarks.landmark

            # Get coordinates for back straightness check
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                        landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                   landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                    landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            
            # Calculate angle for back straightness
            spine_angle = calculate_angle(shoulder, hip, knee)
            
            # Visualize spine angle
            cv2.putText(image, f'Spine Angle: {int(spine_angle)}', 
                        (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
            
            # Get foot positions for feet check
            left_heel_z = landmarks[mp_pose.PoseLandmark.LEFT_HEEL.value].z
            right_heel_z = landmarks[mp_pose.PoseLandmark.RIGHT_HEEL.value].z
            
            # Check if feet are lifting off the ground
            if left_heel_z > 0.25 or right_heel_z > 0.25:  # Adjust threshold as needed
                cv2.putText(image, "Feet not planted!", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(image, "Feet planted", 
                            (50, 100), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2, cv2.LINE_AA)
                       
        except:
            pass
        
        # Render detections
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
        
        cv2.imshow('Squat Form Detection', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
